In [ ]:
import sys
import os

sys.path.insert(0, os.path.abspath("../utils"))
sys.path.insert(0, os.path.abspath("../sdlarch-rl"))


from sdlarch_rl.utils.utils import get_last_index, RealExcludeButtonsWrapper, GenericCNN, TimeLimit, FrameSkip
from sdlarch_rl import make
from stable_baselines3.common.atari_wrappers import WarpFrame
import time
import cv2
import numpy as np
import keyboard
from stable_baselines3.common.atari_wrappers import WarpFrame
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.vec_env import DummyVecEnv
from pathlib import Path
import pygame
from inputs import get_gamepad, devices
import threading
from imitation.data.types import Trajectory
import torch as th
import os
import re
from utils import get_last_index, LSTMWrapper
import gymnasium as gym
from final_fight import FinalFightActionWrapper, make_env
import final_fight
from stable_baselines3.common.policies import ActorCriticPolicy
from stable_baselines3 import PPO

env = make_env()()

# --- CONFIGS ---
MAX_TRAJ = 10
demo_path = 'demos/'
SCALE = 1/4
SCREEN_WIDTH = 854 # int(640 * SCALE)
SCREEN_HEIGHT = 480 # int(480 * SCALE)
NUMBER_OF_ACTIONS = 6
MAX_FPS=30
os.makedirs(demo_path, exist_ok=True)

# bc config
train_path = "./ppo/"
sub_train_path = train_path + "steps"
last_index_imitation = get_last_index(str(sub_train_path), "ppo_policy", "zip")
current_epoch = last_index_imitation


#TODO comment
# current_epoch = 48
latest_model_path = sub_train_path + f"/ppo_policy{current_epoch}.zip"
# policy = ActorCriticPolicy.load(latest_model_path)
model = PPO.load(latest_model_path, env=env)

policy = model.policy

policy = LSTMWrapper(policy)
policy.reset()

class RendererThread(threading.Thread):
    def __init__(self):
        super().__init__(daemon=True)
        self.img = None
        self.color = (0, 0, 255)
        self.count_record = 0
        self.running = True
        self.lock = threading.Lock()
        self.fps = 0

    def update_data(self, img, color, count, fps):
        with self.lock:
            self.img = img
            self.color = color
            self.count_record = count
            self.fps = fps

    def run(self):
        pygame.init()
        window = pygame.display.set_mode((SCREEN_WIDTH, SCREEN_HEIGHT), pygame.HWSURFACE | pygame.DOUBLEBUF)
        pygame.display.set_caption("Final Fight Capture")
        font = pygame.font.SysFont("Arial", 24)
        
        while self.running:
            pygame.event.pump()
            
            img_to_render = None
            with self.lock:
                if self.img is not None:
                    img_to_render = self.img
                    current_color = self.color
                    current_count = self.count_record
                    current_fps = self.fps

            if img_to_render is not None:
                img_rgb = cv2.cvtColor(img_to_render, cv2.COLOR_BGR2RGB)
                surface = pygame.image.frombuffer(img_rgb.flatten(), (SCREEN_WIDTH, SCREEN_HEIGHT), 'RGB')
                
                window.blit(surface, (0, 0))

                
                fps_text = font.render(f"FPS: {int(current_fps)}", True, (255, 255, 255))
                count_text = font.render(f"Demos: {current_count}/{MAX_TRAJ}", True, (255, 255, 255))
                status_color = (0, 255, 0) if current_color == (0, 255, 0) else (255, 0, 0)
                rec_text = font.render("RECORDING" if current_color == (0, 255, 0) else "IDLE", True, status_color)
                
                window.blit(fps_text, (10, 10))
                window.blit(count_text, (10, 40))
                window.blit(rec_text, (10, 70))

                
                pygame.display.flip()
            
            time.sleep(1/30)


last_index = -1
for p in Path(demo_path).iterdir():
    m = re.search(r"demos(\d+)\.pt$", p.name)
    if m: last_index = max(last_index, int(m.group(1)))

gamepad = devices.gamepads[0]
STATE = {k: 0 for k in ["UP", "DOWN", "LEFT", "RIGHT", "A", "B", "X", "START", "CAM_X", "CAM_Y", "LT", "RT", "L3"]}
lock = threading.Lock()

DEADZONE = 10000
NORM = 32768


# Start render thread
renderer = RendererThread()
renderer.start()

is_running = False
is_recording = False
count_record = 0
trajectories = []
recorded_obs = []
recorded_actions = []

print("Done! Press 'K' to start/stop the record.")

fps_avg_frame_count = 0
fps_start_time = time.time()
actual_fps = 0

obs, _ = env.reset()

total_rew = 0

while True:
    loop_start = time.time()
    
    pred_act, _ = policy.predict(obs, deterministic=False)
    
    obs, rew, done, trunc, info = env.step(pred_act)

    total_rew += rew


    if done or trunc:
        print(total_rew)
        env.reset()
        total_rew = 0

    # print(info)
    
    img = env.render()
    if obs is not None:
        # img = obs[0, :, :, -1]
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img_resized =  cv2.resize(img, (SCREEN_WIDTH, SCREEN_HEIGHT), interpolation=cv2.INTER_NEAREST)
        color = (0, 255, 0) if is_running else (0, 0, 255)
        renderer.update_data(img_resized, color, count_record, actual_fps)

    
    fps_avg_frame_count += 1
    if time.time() - fps_start_time >= 1.0: # update fps every 1 second
        actual_fps = fps_avg_frame_count / (time.time() - fps_start_time)
        fps_avg_frame_count = 0
        fps_start_time = time.time()
        # print(f"FPS of capture: {actual_fps:.2f}")

    # try fix fps (25)
    time_to_wait = (1/40) - (time.time() - loop_start)
    if time_to_wait > 0:
        time.sleep(time_to_wait)

env.close()
renderer.running = False
pygame.quit()

D:\Python311\Lib\site-packages\pygame\pkgdata.py:25: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
  from pkg_resources import resource_stream, resource_exists


statename is None setting to default state


D:\Python311\Lib\site-packages\stable_baselines3\common\save_util.py:449: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  th_object = th.load(file_content, map_location=device

Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
Wrapping the env in a VecTransposeImage.
Done! Press 'K' to start/stop the record.
-129.39200000000028
